In [ ]:
import sys
print(sys.executable)

In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns 
import numpy as np
import json
import lightgbm as lgb

from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import classification_report
from sklearn.metrics import recall_score,average_precision_score

In [ ]:
# read bigquery data into pandas dataframe
import pandas as pd

df = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.cafe.cafe-sales`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
    location="australia-southeast1",
)

In [ ]:
# Drop the unnecessary columns
df.drop(columns=['ip_addr', 'date_paid', 'status', 'items'], inplace=True)

In [ ]:
df.info()

### Create customer infomation table

- period: 2019-2020.12.31, 2021-2021.12.31, 2022-2024

In [ ]:
def create_rfm_table(df, date_start, date_end, period=60, test=False):
    """
    Create the rfm table based of the given period of time.
    Churn: 1 for no transaction made in the observation period, vice versa

    :param date_start: The starting date of the dataset
    :param date_end: The ending date of the dataset
    :param period: Observation period, default 60 days
    :return: Return the rfm table created based on the period dataset
    """ 
    
    period_start = date_end - pd.DateOffset(days=period)

    table = df.copy()
    table.set_index('date_created', inplace=True) # setting Date as index
    table.sort_index(inplace=True)

    df_part1 = table.loc[date_start:period_start] # part of data that doesn't have last 60 days of transaction
    df_part1_obs = table.loc[period_start:date_end] # part of data that have only last 60 days of transaction

    # reseting the index
    df_part1.reset_index(inplace=True)
    df_part1_obs.reset_index(inplace=True)

    ## calculate the rfm
    rfm_df = df_part1.groupby('customer_id').agg(
        recency=('date_created', lambda x: (df_part1['date_created'].max() - x.max()).days),
        frequency=('date_created', 'count'),
        monetary=('total', 'sum')
    ).reset_index()

    # getting the number of customers in part1 and part2
    part1_customer = df_part1['customer_id'].sort_values().unique()
    part1_obs_customer = df_part1_obs['customer_id'].sort_values().unique()

    # finding whether the customer is churned or not
    rfm_df['churn'] = [0 if customer in part1_obs_customer else 1 for customer in part1_customer]

    # drop the customer_id as unnecessary
    if test == False:
        rfm_df.drop(['customer_id'], axis= 1, inplace= True)

    return rfm_df

In [ ]:
first_order_dates = df['date_created'].min()
period_1 = pd.Timestamp('2020-12-31')
period_2 = pd.Timestamp('2021-12-31')
last_order_dates = df['date_created'].max()

rfm_part1 = create_rfm_table(df, first_order_dates, period_1)
rfm_part2 = create_rfm_table(df, period_1, period_2)
rfm_part3 = create_rfm_table(df, period_2, last_order_dates)

rfm_part4 = create_rfm_table(df, first_order_dates, period_2)
rfm_part5 = create_rfm_table(df, period_1, last_order_dates)

rfm_part6 = create_rfm_table(df, first_order_dates, first_order_dates + pd.DateOffset(days=365))
rfm_part7 = create_rfm_table(df, last_order_dates - pd.DateOffset(days=365), last_order_dates)
rfm_part8 = create_rfm_table(df, first_order_dates + pd.DateOffset(days=365), last_order_dates - pd.DateOffset(days=365))

test_set = create_rfm_table(df, first_order_dates, last_order_dates, test=True)

rfm_table = pd.concat([rfm_part1, rfm_part2,rfm_part3, rfm_part4, rfm_part5, rfm_part6, rfm_part7, rfm_part8], axis=0)

In [ ]:
rfm_table = rfm_table[rfm_table['recency'] < 40]
test_set = test_set[test_set['recency'] < 40]

In [ ]:
rfm_table.churn.value_counts()

In [ ]:
test_set.churn.value_counts()

### Model

#### Data standardization & Data splitting

In [ ]:
X = rfm_table.drop(['churn'], axis= 1)
y = rfm_table[['churn']].values.reshape(-1)

In [ ]:
id_test = test_set['customer_id']
X_test = test_set.drop(['churn', 'customer_id'], axis= 1)
y_test = test_set[['churn']].values.reshape(-1)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit and Transform The Data
X = scaler.fit_transform(X)
X_test = scaler.fit_transform(X_test)

In [ ]:
from sklearn.model_selection import train_test_split

test_size = 0.2
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=test_size, random_state=42, shuffle=True)

#### LGB

In [ ]:
import lightgbm as lgb

In [ ]:
params = {
'objective': 'binary',
'boosting_type': 'gbdt',
'metric':'auc',
'num_leaves': 31,
'learning_rate': 0.05,
'scale_pos_weight': 0.5,
"verbose": 0,
}

lgb_train = lgb.Dataset(X_train, y_train)
lgb_eval = lgb.Dataset(X_val, y_val, reference=lgb_train)

clf = lgb.train(params,
                lgb_train, 
                valid_sets=[lgb_train, lgb_eval],
                num_boost_round=20,
                callbacks=[lgb.early_stopping(stopping_rounds=10),])

In [ ]:
y_pred_proba = clf.predict(X_test, num_iteration=clf.best_iteration)

y_pred = (y_pred_proba > 0.5).astype("int")
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
    
print('The accuracy of prediction is:', accuracy)
print('The ROC AUC of prediction is:', roc_auc)
print('The F1 Score of prediction is:', f1)
print('The Precision of prediction is:', prec)
print('The Recall of prediction is:', rec)

In [ ]:
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

plt.plot([0,1],[0,1],'k--')
plt.plot(fpr,tpr, label='LGBM')
plt.xlabel('fpr')
plt.ylabel('tpr')
plt.title(f'LGBM ROC curve')
plt.show()

In [ ]:
lgb.plot_importance(clf, figsize=(10, 6))

#### LGBMClassifier --- Used for finding valuable customers

In [ ]:
params = {
'objective': 'binary',
'boosting_type': 'gbdt',
'metric':'auc',
'num_leaves': 30,
"max_depth": 7,
'learning_rate': 0.01,
'scale_pos_weight': 0.5,
"verbose": 1,
}

clf_binary = lgb.LGBMClassifier(**params).fit(X_train, y_train,eval_set=[(X_val,y_val),(X_train,y_train)])

In [ ]:
predictions = clf_binary.predict_proba(X_test)


acc_bin  = accuracy_score(y_test, predictions[:,1] >= 0.5)
roc_auc_bin   = roc_auc_score(y_test, predictions[:,1])
f1_bin = f1_score(y_test, predictions[:,1] >= 0.5)
prec_bin = precision_score(y_test, predictions[:,1] >= 0.5)
rec_bin    = recall_score(y_test, predictions[:,1] >= 0.5)

print('The accuracy of prediction is:', acc_bin)
print('The ROC AUC of prediction is:', roc_auc_bin)
print('The F1 Score of prediction is:', f1_bin)
print('The Precision of prediction is:', prec_bin)
print('The Recall of prediction is:', rec_bin)

In [ ]:
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_test, predictions[:,1])

plt.plot([0,1],[0,1],'k--')
plt.plot(fpr,tpr, label='LGBM-Binary')
plt.xlabel('fpr')
plt.ylabel('tpr')
plt.title(f'LGBM-Binary ROC curve')
plt.show()

In [ ]:
lgb.plot_importance(clf_binary, figsize=(10, 6))

In [ ]:
print(classification_report(y_test, predictions[:,1] >= 0.5))

##### Finding valuable customers

In [ ]:
rfm_df = pd.read_gbq(
    """
    SELECT  *
    FROM `jr-data-training.nicole.rfm_new`
    --- LIMIT 10
  """,
    project_id="jr-data-training",
)

过去消费 * churn rate = expected value lost -> 做排序top多少 找出最有价值的客户（）


筛选条件：
- 60天内无回购 （churn = 1）
- prediction 概率 >= 0.5
- 消费过3次以上 （frequency >= 3）

In [ ]:
test_set.reset_index(drop=True, inplace=True)
predictions_df = pd.DataFrame({'prediction': predictions[:,1]})
merged_df = pd.concat([test_set, predictions_df], axis=1)

# calculate the expected value lost
merged_df["expected_value_lost"] = merged_df.monetary * merged_df.prediction

# sort the lost customers by the value lost
sorted_df = merged_df[(merged_df['churn'] == 1) & (merged_df['prediction'] >= 0.5) & (merged_df['frequency'] >= 2)].sort_values(by='expected_value_lost', ascending=False)

# merge the RFM segment value into the table
sorted_df = pd.merge(sorted_df, rfm_df[['customer_id', 'Segment']], on='customer_id', how='left')


原始 churn rate：

- 73 / (153 + 73) ≈ 0.322

假设挽回了这31个人，修改后的 churn rate：
- (73 - 31) / (153 + 73 - 31) ≈ 0.215

变化百分比：
- (0.215 - 0.322) / 0.322 × 100% ≈ -33.23%

挽回金额：
- 2459.48

挽回金额百分比：
- 2459.48 / 18222.09 × 100% ≈ 13.50%


In [ ]:
import pandas_gbq
pandas_gbq.to_gbq(sorted_df, 'jr-data-training.nicole.lost_customers', project_id="jr-data-training")

#### RANDOM FOREST

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=20).fit(X_train, y_train)
#rf_over = RandomForestClassifier(n_estimators=20).fit(X_train_over, y_train_over)

In [ ]:
predictions = pd.DataFrame()
predictions['true'] = y_train
predictions['preds'] = rf.predict(X_train)

In [ ]:
predictions_test = pd.DataFrame()
predictions_test['true'] = y_test
predictions_test['preds'] = rf.predict(X_test)
predictions_test['proba'] = rf.predict_proba(X_test)[:, 1]
#predictions_test['preds_over'] = rf_over.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
train_acc = accuracy_score(predictions.true, predictions.preds)
test_acc = accuracy_score(predictions_test.true, predictions_test.preds)
#test_acc_over = accuracy_score(predictions_test.true, predictions_test.preds_over)

roc_auc_rf   = roc_auc_score(predictions_test.true, predictions_test.proba)

print('The ROC AUC of prediction is:', roc_auc_rf)
print(f"Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")

In [ ]:
print(classification_report(predictions_test.true, predictions_test.preds))

In [ ]:
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(predictions_test.true, predictions_test.proba)

plt.plot([0,1],[0,1],'k--')
plt.plot(fpr,tpr, label='Random Forest')
plt.xlabel('fpr')
plt.ylabel('tpr')
plt.title(f'Random Forest ROC curve')
plt.show()

In [ ]:
current_rfm = rfm_table.copy()
current_rfm.drop(columns=['churn'], inplace=True)
probs = rf.predict_proba(current_rfm)[:, 1]

plt.figure(figsize=(12, 6))
# matplotlib histogram
plt.hist(probs, bins = int(180/6))

plt.title('Probability Distribution of Churn Risk')
plt.xlabel('Churn Risk')
plt.ylabel('# Customers')
plt.show()